# AIPP Voice Trainer (Colab): latih model suara RVC milik Anda
Notebook ini melatih model RVC dari rekaman suara Anda sendiri memakai **Applio**. Hasilnya berupa `Nama.zip` (berisi `.pth` + `.index`) yang bisa langsung dipakai di notebook **AIPP Voice Studio**.

## Siapkan rekaman dulu (ini penentu kualitas)
- **Durasi:** 30 detik tidak cukup. Minimal 5 menit, **15-30 menit** untuk hasil bagus. Makin banyak dan bervariasi, makin stabil.
- **Tempat:** ruangan tenang dan tidak bergema (kamar berkarpet, dekat lemari pakaian atau bantal). Matikan kipas dan AC. Tanpa musik latar.
- **Mikrofon:** satu mikrofon yang konsisten, jarak mulut sekitar 10-15 cm, volume tidak sampai pecah (clipping). Simpan sebagai WAV bila bisa.
- **Isi:** bacakan teks yang beragam dengan ekspresi: berita, cerita, kalimat tanya dan seru, angka, nada santai dan serius. Rekaman datar menghasilkan suara datar.
- **Kalau ingin dipakai untuk cover lagu:** sertakan 5-10 menit Anda menyanyi (tanpa musik) dengan jangkauan nada yang bervariasi. Model dari rekaman bicara saja sering goyang saat menyanyi.
- Tidak perlu transkrip. Gunakan hanya suara Anda sendiri.

## Cara pakai
1. *Runtime > Change runtime type > T4 GPU*.
2. Jalankan sel 1 sampai 9 berurutan. Sel 4 memeriksa kualitas rekaman Anda dan memberi peringatan; perbaiki dulu kalau ada masalah.
3. Sel 6 (training) paling lama. Checkpoint tersimpan di Google Drive (`AIPP_RVC/logs`), jadi bila Colab terputus Anda bisa melanjutkan.
4. Sel 8 membandingkan beberapa checkpoint. Dengarkan sendiri dan pilih yang paling natural, lalu isi `CHOSEN_EPOCH` di sel 9.

## Catatan jujur
- Perintah di notebook ini saya cocokkan dengan kode sumber Applio (commit `6d5d951`), tetapi **proses training belum saya jalankan sampai selesai** karena butuh GPU dan unduhan dari luar. Kalau ada error, kirim pesannya.
- Lama training bergantung pada GPU dan jumlah data; jangan berasumsi, pantau lognya.
- Epoch yang berlebihan membuat suara pecah (overtrain). Itu sebabnya sel 8 membandingkan beberapa checkpoint.
- Hasil tidak dijanjikan sempurna. Kualitas mengikuti kualitas rekaman.
- Vocoder dibiarkan HiFi-GAN agar model kompatibel dengan notebook AIPP Voice Studio.


In [ ]:
# SEL 1 - Pengaturan, GPU, dan Google Drive (checkpoint disimpan di Drive agar training bisa dilanjutkan bila Colab terputus)
MODEL_NAME = "Irwin_v3"       # tanpa spasi atau simbol
SAMPLE_RATE = 40000           # 40000 = seimbang (disarankan). 48000 = kualitas lebih tinggi tapi lebih berat
TOTAL_EPOCH = 300             # mulai 300; jangan berlebihan (overtrain membuat suara pecah)
SAVE_EVERY = 25               # simpan bobot tiap N epoch supaya bisa membandingkan checkpoint
BATCH_SIZE = 8                # naikkan ke 12 bila memori GPU lega, turunkan ke 4 bila out-of-memory
F0_METHOD = "rmvpe"
VOCODER = "HiFi-GAN"          # tetap HiFi-GAN agar model cocok dengan notebook AIPP Voice Studio
SAVE_TO_DRIVE = True

import os, torch
assert torch.cuda.is_available(), "GPU tidak aktif. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang."
print("GPU:", torch.cuda.get_device_name(0))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/AIPP_RVC"
else:
    BASE = "/content/AIPP_RVC"
LOGS = BASE + "/logs"; HASIL = BASE + "/hasil"
os.makedirs(LOGS, exist_ok=True); os.makedirs(HASIL, exist_ok=True)
os.environ["APPLIO_LOGS_DIR"] = LOGS   # dibaca oleh Applio
print("Folder kerja:", BASE)


In [ ]:
# SEL 2 - Pasang Applio (versi dikunci agar perintah di notebook ini cocok). Butuh beberapa menit.
!apt-get -qq install -y ffmpeg libportaudio2 > /dev/null
!rm -rf /content/Applio && git clone -q https://github.com/IAHispano/Applio.git /content/Applio && cd /content/Applio && git checkout -q 6d5d951
!cd /content/Applio && pip install -q -r requirements-engine.txt
!cd /content/Applio && python core.py prerequisites
# Jika muncul error versi paket/numpy: Runtime > Restart session, lalu jalankan ulang dari sel 1.


In [ ]:
# SEL 3 - Masukkan rekaman suara Anda (WAV/MP3/M4A/FLAC, atau satu file .zip berisi banyak rekaman)
# Opsi B: isi DATA_DRIVE dengan folder di Drive, mis. "/content/drive/MyDrive/rekaman_irwin"
DATA_DRIVE = ""

import os, glob, shutil, zipfile, subprocess
RAW, CLEAN = "/content/dataset/raw", "/content/dataset/clean"
for d in (RAW, CLEAN):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
if DATA_DRIVE:
    shutil.copytree(DATA_DRIVE, RAW, dirs_exist_ok=True)
else:
    from google.colab import files
    for name in files.upload():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as z: z.extractall(RAW)
        else:
            shutil.move(name, os.path.join(RAW, name))

EXT = (".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".opus", ".webm", ".wma")
src = [p for p in glob.glob(RAW + "/**/*", recursive=True) if p.lower().endswith(EXT)]
assert src, "Tidak ada file audio yang ditemukan."
for i, p in enumerate(sorted(src)):
    out = os.path.join(CLEAN, "rec_%03d.wav" % i)
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", p, "-ac", "1", "-ar", "44100", "-c:a", "pcm_s16le", out], check=True)
print("Dikonversi ke WAV mono 44,1 kHz:", len(src), "file ->", CLEAN)


In [ ]:
# SEL 4 - Cek kualitas rekaman SEBELUM training (angka nyata dari file Anda, bukan tebakan)
import glob, numpy as np, soundfile as sf

def report(folder):
    tot, rows, warn = 0.0, [], []
    for p in sorted(glob.glob(folder + "/*.wav")):
        x, sr = sf.read(p, dtype="float32")
        if x.ndim > 1: x = x.mean(1)
        dur = len(x) / sr; tot += dur
        hop = int(sr * 0.025); n = len(x) // hop
        if n < 4: continue
        fr = x[: n * hop].reshape(n, hop)
        rms = 20 * np.log10(np.sqrt((fr ** 2).mean(1)) + 1e-9)
        noise, speech = np.percentile(rms, 10), np.percentile(rms, 90)
        clip = float((np.abs(x) >= 0.999).mean() * 100)
        rows.append((p.split("/")[-1], dur, speech - noise, noise, clip))
    print("%-14s %7s %10s %10s %8s" % ("file", "detik", "SNR~ dB", "noise dBFS", "clip %"))
    for r in rows: print("%-14s %7.1f %10.1f %10.1f %8.2f" % r)
    mins = tot / 60
    print("\nTotal durasi: %.1f menit dari %d file" % (mins, len(rows)))
    if mins < 5: warn.append("Kurang dari 5 menit: hasil hampir pasti tidak stabil. Tambah rekaman.")
    elif mins < 10: warn.append("5-10 menit: cukup untuk uji coba, tapi 15-30 menit jauh lebih baik.")
    elif mins <= 30: print("Durasi bagus (10-30 menit).")
    else: print("Durasi sangat baik (>30 menit).")
    snr = np.array([r[2] for r in rows]); noise = np.array([r[3] for r in rows]); clip = np.array([r[4] for r in rows])
    if len(rows) and np.median(snr) < 25: warn.append("Rekaman cenderung bising/bergema (selisih suara-vs-latar median %.0f dB, idealnya >30). Rekam ulang di ruangan lebih tenang." % np.median(snr))
    if len(rows) and (clip > 0.1).any(): warn.append("Ada file dengan clipping (suara pecah di rekaman). Jauhkan mikrofon atau turunkan gain, lalu rekam ulang file itu.")
    if len(rows) and np.median(noise) > -45: warn.append("Lantai noise tinggi (median %.0f dBFS). Matikan kipas/AC dan jauhkan sumber bising." % np.median(noise))
    for w in warn: print("PERINGATAN:", w)
    if not warn: print("Tidak ada masalah terdeteksi oleh pemeriksaan otomatis. Tetap dengarkan beberapa file sendiri.")
    return mins

DUR_MIN = report(CLEAN)


In [ ]:
# SEL 5 - Preprocess (potong otomatis, filter rumble) dan ekstraksi fitur
!cd /content/Applio && python core.py preprocess --model-name "{MODEL_NAME}" --dataset-path /content/dataset/clean --sample-rate {SAMPLE_RATE} --cut-preprocess Automatic --process-effects --chunk-len 3.0 --overlap-len 0.3
!cd /content/Applio && python core.py extract --model-name "{MODEL_NAME}" --f0-method {F0_METHOD} --sample-rate {SAMPLE_RATE} --gpu 0 --embedder-model contentvec


In [ ]:
# SEL 6 - TRAINING. Berjalan lama dan menampilkan log; jangan tutup tab.
# Bila Colab terputus: jalankan ulang sel 1, 2, lalu sel ini. Training dilanjutkan dari checkpoint terakhir di Drive (jangan jalankan ulang sel 3 dan 5).
!cd /content/Applio && python core.py train --model-name "{MODEL_NAME}" --vocoder "{VOCODER}" --save-every-epoch {SAVE_EVERY} --save-every-weights --total-epoch {TOTAL_EPOCH} --sample-rate {SAMPLE_RATE} --batch-size {BATCH_SIZE} --gpu 0 --pretrained

In [ ]:
# SEL 7 - Buat file index (biasanya sudah otomatis di akhir training; jalankan sel ini bila training terputus atau file index belum ada)
import os
if not os.path.exists(f"{LOGS}/{MODEL_NAME}/{MODEL_NAME}.index"):
    !cd /content/Applio && python core.py index --model-name "{MODEL_NAME}"
print("Index:", "ADA" if os.path.exists(f"{LOGS}/{MODEL_NAME}/{MODEL_NAME}.index") else "BELUM ADA")

In [ ]:
# SEL 7b - Ekstrak bobot inferensi dari checkpoint G_ yang sudah ada (tanpa training ulang)
import os, sys, glob, re, json, torch
os.chdir("/content/Applio"); sys.path.insert(0, "/content/Applio")
from rvc.train.process.extract_model import extract_model
from rvc.train.utils import HParams

mdir = f"{LOGS}/{MODEL_NAME}"
with open(f"{mdir}/config.json", "r", encoding="utf-8") as f:
    hps = HParams(**json.load(f))

STEP_PER_EPOCH = None   # None = dihitung otomatis dari nama file G_ dan epoch di dalamnya

for g in sorted(glob.glob(f"{mdir}/G_*.pth"), key=lambda p: int(re.search(r"G_(\d+)", p).group(1))):
    step = int(re.search(r"G_(\d+)", g).group(1))
    if step == 2333333:
        continue
    ck = torch.load(g, map_location="cpu", weights_only=True)
    epoch = int(ck["iteration"])
    target = f"{mdir}/{MODEL_NAME}_{epoch}e_{step}s.pth"
    if os.path.exists(target):
        print("Sudah ada:", os.path.basename(target)); continue
    hps.model.spk_embed_dim = ck["model"]["emb_g.weight"].shape[0]
    extract_model(ckpt=ck["model"], sr=hps.data.sample_rate, name=MODEL_NAME, model_path=target,
                  epoch=epoch, step=step, hps=hps, vocoder=VOCODER)
    del ck

print(sorted(os.path.basename(p) for p in glob.glob(f"{mdir}/{MODEL_NAME}_*e_*s.pth")))

In [ ]:
# SEL 8 - Uji dengar: bandingkan beberapa checkpoint pada audio yang sama, lalu pilih yang paling natural
# Terlalu banyak epoch bisa membuat suara pecah/serak, jadi checkpoint terakhir belum tentu terbaik.
import os, re, glob, subprocess
from IPython.display import Audio, display
TEST_AUDIO = ""   # kosongkan untuk mengunggah satu rekaman uji (suara apa pun; idealnya Anda berbicara/bernyanyi dengan ekspresi)
if not TEST_AUDIO:
    from google.colab import files
    TEST_AUDIO = list(files.upload().keys())[0]
# Ubah apa pun (mp3, m4a, wav, dll.) menjadi WAV mono agar pasti terbaca
os.makedirs("/content/uji", exist_ok=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", TEST_AUDIO, "-ac", "1", "-ar", "44100", "/content/uji/test.wav"], check=True)
TEST_AUDIO = "/content/uji/test.wav"

mdir = f"{LOGS}/{MODEL_NAME}"
ckpts = sorted(glob.glob(f"{mdir}/{MODEL_NAME}_*e_*s.pth"), key=lambda p: int(re.search(r"_(\d+)e_", p).group(1)))
assert ckpts, "Belum ada checkpoint bobot (.pth). Jalankan training dulu."
idx = f"{mdir}/{MODEL_NAME}.index"
assert os.path.exists(idx), "File index belum ada. Jalankan sel 7."
pick = ckpts[-6:]   # 6 checkpoint terakhir
for c in pick:
    ep = re.search(r"_(\d+)e_", c).group(1)
    out = f"/content/uji/epoch_{ep}.wav"
    subprocess.run(["python", "core.py", "infer", "--input-path", TEST_AUDIO, "--output-path", out, "--pth-path", c, "--index-path", idx], cwd="/content/Applio", check=True)
    print("Epoch", ep); display(Audio(out))

In [ ]:
# SEL 9 - Ekspor model terbaik (.pth + .index) dalam satu zip yang bisa dipakai di notebook AIPP Voice Studio
CHOSEN_EPOCH = None   # isi angka epoch pilihan Anda dari sel 8, mis. 225. None = checkpoint terakhir
import os, re, glob, shutil, zipfile
mdir = f"{LOGS}/{MODEL_NAME}"
ckpts = {int(re.search(r"_(\d+)e_", p).group(1)): p for p in glob.glob(f"{mdir}/{MODEL_NAME}_*e_*s.pth")}
assert ckpts, "Tidak ada checkpoint."
ep = CHOSEN_EPOCH if CHOSEN_EPOCH else max(ckpts)
assert ep in ckpts, "Epoch %s tidak ada. Tersedia: %s" % (ep, sorted(ckpts))
out = f"{HASIL}/{MODEL_NAME}"; os.makedirs(out, exist_ok=True)
pth = f"{out}/{MODEL_NAME}.pth"; idx = f"{out}/added_{MODEL_NAME}.index"
shutil.copy(ckpts[ep], pth); shutil.copy(f"{mdir}/{MODEL_NAME}.index", idx)
zp = f"{HASIL}/{MODEL_NAME}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pth, f"{MODEL_NAME}/{MODEL_NAME}.pth"); z.write(idx, f"{MODEL_NAME}/added_{MODEL_NAME}.index")
print("Epoch dipilih:", ep, "| tersimpan di Drive:", zp, "(%.0f MB)" % (os.path.getsize(zp) / 1e6))
try:
    from google.colab import files; files.download(zp)
except Exception: pass
